# Phase 3: Multi-Label Category Classification (BERT Binary Relevance)

## First Multi-Label Task: 85 BGG Categories from Text Descriptions

**Key Differences from Phase 1-2 (Single-Label Geek Type):**
- **Loss**: BCEWithLogitsLoss (independent binary classification per label)
- **Encoding**: Multi-hot vectors [N, 85] instead of class indices
- **Splitting**: iterative_train_test_split (multi-label stratification)
- **Metrics**: Micro-F1, Macro-F1, LRAP, Hamming Loss, Subset Accuracy
- **Threshold**: Optimized on validation set (not fixed at 0.5)
- **Output**: Sigmoid per label (not softmax across classes)

**Dataset**: 165,709 games, 85 categories, avg 2.74 labels/game, imbalance ratio 792.6

**Research Questions Addressed:**
- **RQ1**: Can text descriptions identify game categories? (BERT-BR baseline)
- **RQ3**: How do text characteristics affect per-label performance?

---

## Interactive Navigation

- **[Section 0](#section-0)** - Data Loading & Cleaning
- **[Section 1](#section-1)** - Multi-Label Encoding & Stratified Splitting
- **[Section 2](#section-2)** - TF-IDF Binary Relevance Baseline
- **[Section 3](#section-3)** - BERT Binary Relevance Model
- **[Section 4](#section-4)** - Training Loop (Multi-Label)
- **[Section 5](#section-5)** - Threshold Optimization
- **[Section 6](#section-6)** - Results & Per-Label Analysis
- **[Section 7](#section-7)** - Error Analysis & Visualization

---


# SECTION 0: Data Loading & Cleaning {#section-0}

Load category subset, drop empty descriptions, verify data quality.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from itertools import chain
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import time
import json
import pickle
warnings.filterwarnings("ignore")

# === Reproducibility ===
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# === Device ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
print()

# === Load Dataset ===
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/My Drive/Notebooks Tese/Dataset')
else:
    DATA_DIR = Path(r"C:\Users\marco\OneDrive\Ambiente de Trabalho\Tese\Dataset\XML Dataset")

print(f'Data directory: {DATA_DIR}')

df = pd.read_parquet(DATA_DIR / "bgg_category_subset.parquet")
print(f'Raw dataset: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

# === Clean: drop empty descriptions ===
empty_mask = df['description_clean'].str.strip() == ''
print(f'Empty descriptions found: {empty_mask.sum()} (dropping)')
df = df[~empty_mask].reset_index(drop=True)
print(f'Clean dataset: {df.shape}')
print()

# === Quick verification ===
print('Sample entries:')
for i in range(3):
    print(f'  {df.iloc[i]["name"]}: {df.iloc[i]["categories_list"]}')


# SECTION 1: Multi-Label Encoding & Stratified Splitting {#section-1}

**Critical differences from single-label:**
- Labels encoded as multi-hot binary matrix [N, 85]
- Stratification uses iterative_train_test_split from skmultilearn (preserves label co-occurrence distributions)
- Verification checks per-label frequency across splits


In [ ]:
# === Multi-Hot Encoding ===
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y_multilabel = mlb.fit_transform(df['categories_list'])

label_names = mlb.classes_
num_labels = len(label_names)

print(f'Multi-hot matrix shape: {y_multilabel.shape}')
print(f'Number of labels: {num_labels}')
print(f'Avg labels per game: {y_multilabel.sum(axis=1).mean():.2f}')
print(f'Label frequency range: [{y_multilabel.sum(axis=0).min()}, {y_multilabel.sum(axis=0).max()}]')
print()

# Label frequency table
label_freq = pd.DataFrame({
    'Label': label_names,
    'Count': y_multilabel.sum(axis=0),
    'Percentage': (y_multilabel.sum(axis=0) / len(df) * 100).round(2)
}).sort_values('Count', ascending=False).reset_index(drop=True)

print('Top 10 most frequent categories:')
print(label_freq.head(10).to_string(index=False))
print()
print('Bottom 10 least frequent categories:')
print(label_freq.tail(10).to_string(index=False))


In [ ]:
# === Multi-Label Stratified Split ===
# iterative_train_test_split preserves label distributions better than random split

try:
    from skmultilearn.model_selection import iterative_train_test_split
    USE_ITERATIVE = True
    print('Using iterative_train_test_split (skmultilearn)')
except ImportError:
    from sklearn.model_selection import train_test_split
    USE_ITERATIVE = False
    print('WARNING: skmultilearn not available, falling back to sklearn train_test_split')
    print('  Install with: pip install scikit-multilearn')
    print('  This may result in suboptimal label distribution across splits')

X_data = df[['id', 'name', 'description_clean']].values
y_data = y_multilabel

if USE_ITERATIVE:
    # Split 1: Train (70%) vs Temp (30%)
    X_train, y_train, X_temp, y_temp = iterative_train_test_split(
        X_data, y_data, test_size=0.30
    )
    # Split 2: Val (15%) vs Test (15%)  
    X_val, y_val, X_test, y_test = iterative_train_test_split(
        X_temp, y_temp, test_size=0.50
    )
else:
    indices = np.arange(len(X_data))
    train_idx, temp_idx = train_test_split(indices, test_size=0.30, random_state=SEED)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=SEED)
    X_train, y_train = X_data[train_idx], y_data[train_idx]
    X_val, y_val = X_data[val_idx], y_data[val_idx]
    X_test, y_test = X_data[test_idx], y_data[test_idx]

# Convert to DataFrames for text access
X_train_df = pd.DataFrame(X_train, columns=['id', 'name', 'description_clean'])
X_val_df = pd.DataFrame(X_val, columns=['id', 'name', 'description_clean'])
X_test_df = pd.DataFrame(X_test, columns=['id', 'name', 'description_clean'])

print(f'\nData splits:')
print(f'  Train: {len(X_train_df):,} ({len(X_train_df)/len(df):.1%})')
print(f'  Val:   {len(X_val_df):,} ({len(X_val_df)/len(df):.1%})')
print(f'  Test:  {len(X_test_df):,} ({len(X_test_df)/len(df):.1%})')


In [ ]:
# === Verify Multi-Label Stratification ===

train_dist = y_train.mean(axis=0)
val_dist = y_val.mean(axis=0)
test_dist = y_test.mean(axis=0)

max_diff_val = np.abs(train_dist - val_dist).max()
max_diff_test = np.abs(train_dist - test_dist).max()
mean_diff_val = np.abs(train_dist - val_dist).mean()
mean_diff_test = np.abs(train_dist - test_dist).mean()

print('=== Multi-Label Stratification Verification ===')
print(f'  Max label freq diff (train vs val):  {max_diff_val:.4f}')
print(f'  Max label freq diff (train vs test): {max_diff_test:.4f}')
print(f'  Mean label freq diff (train vs val): {mean_diff_val:.4f}')
print(f'  Mean label freq diff (train vs test): {mean_diff_test:.4f}')

if max_diff_val < 0.02 and max_diff_test < 0.02:
    print('  Status: PASSED (all labels within 2% tolerance)')
else:
    worst_label_val = label_names[np.abs(train_dist - val_dist).argmax()]
    worst_label_test = label_names[np.abs(train_dist - test_dist).argmax()]
    print(f'  Status: WARNING - Some labels exceed 2% tolerance')
    print(f'  Worst (train vs val): {worst_label_val} (diff={max_diff_val:.4f})')
    print(f'  Worst (train vs test): {worst_label_test} (diff={max_diff_test:.4f})')
print()

# Cardinality distribution check
train_card = y_train.sum(axis=1).mean()
val_card = y_val.sum(axis=1).mean()
test_card = y_test.sum(axis=1).mean()
print(f'Avg cardinality: Train={train_card:.2f}, Val={val_card:.2f}, Test={test_card:.2f}')


# SECTION 2: TF-IDF Binary Relevance Baseline {#section-2}

Classical ML baseline: TF-IDF features + independent Logistic Regression per label.
This establishes the lower bound for transformer comparison.


In [ ]:
# === TF-IDF + Binary Relevance (OneVsRest) Baseline ===
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, hamming_loss, label_ranking_average_precision_score

print('='*80)
print('BASELINE: TF-IDF + Logistic Regression (Binary Relevance)')
print('='*80)
print()

# Vectorize
print('Vectorizing text...')
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5)
X_train_tfidf = vectorizer.fit_transform(X_train_df['description_clean'].values)
X_val_tfidf = vectorizer.transform(X_val_df['description_clean'].values)
X_test_tfidf = vectorizer.transform(X_test_df['description_clean'].values)
print(f'TF-IDF shape: {X_train_tfidf.shape}')

# Train Binary Relevance (OneVsRest with LogReg)
print('Training OneVsRest Logistic Regression (85 classifiers)...')
start_time = time.time()
ovr_model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, C=1.0, random_state=SEED),
    n_jobs=-1
)
ovr_model.fit(X_train_tfidf, y_train)
train_time = time.time() - start_time
print(f'Training time: {train_time:.1f}s')

# Predict (binary) and predict_proba (for LRAP)
y_val_pred_tfidf = ovr_model.predict(X_val_tfidf)
y_test_pred_tfidf = ovr_model.predict(X_test_tfidf)

# Get probabilities for LRAP
y_val_proba_tfidf = ovr_model.predict_proba(X_val_tfidf) if hasattr(ovr_model, 'predict_proba') else None
y_test_proba_tfidf = ovr_model.predict_proba(X_test_tfidf) if hasattr(ovr_model, 'predict_proba') else None

# === Multi-Label Metrics ===
print()
print('VALIDATION SET:')
val_micro_f1 = f1_score(y_val, y_val_pred_tfidf, average='micro', zero_division=0)
val_macro_f1 = f1_score(y_val, y_val_pred_tfidf, average='macro', zero_division=0)
val_hamming = hamming_loss(y_val, y_val_pred_tfidf)
val_subset_acc = (y_val == y_val_pred_tfidf).all(axis=1).mean()
val_lrap = label_ranking_average_precision_score(y_val, y_val_proba_tfidf) if y_val_proba_tfidf is not None else 0

print(f'  Micro-F1:      {val_micro_f1:.4f}')
print(f'  Macro-F1:      {val_macro_f1:.4f}')
print(f'  Hamming Loss:  {val_hamming:.4f}')
print(f'  Subset Acc:    {val_subset_acc:.4f}')
print(f'  LRAP:          {val_lrap:.4f}')

print()
print('TEST SET:')
test_micro_f1_tfidf = f1_score(y_test, y_test_pred_tfidf, average='micro', zero_division=0)
test_macro_f1_tfidf = f1_score(y_test, y_test_pred_tfidf, average='macro', zero_division=0)
test_hamming_tfidf = hamming_loss(y_test, y_test_pred_tfidf)
test_subset_acc_tfidf = (y_test == y_test_pred_tfidf).all(axis=1).mean()
test_lrap_tfidf = label_ranking_average_precision_score(y_test, y_test_proba_tfidf) if y_test_proba_tfidf is not None else 0

print(f'  Micro-F1:      {test_micro_f1_tfidf:.4f}')
print(f'  Macro-F1:      {test_macro_f1_tfidf:.4f}')
print(f'  Hamming Loss:  {test_hamming_tfidf:.4f}')
print(f'  Subset Acc:    {test_subset_acc_tfidf:.4f}')
print(f'  LRAP:          {test_lrap_tfidf:.4f}')

# Store baseline results
tfidf_results = {
    'micro_f1': test_micro_f1_tfidf,
    'macro_f1': test_macro_f1_tfidf,
    'hamming': test_hamming_tfidf,
    'subset_acc': test_subset_acc_tfidf,
    'lrap': test_lrap_tfidf
}
print('\nTF-IDF baseline complete')


# SECTION 3: BERT Binary Relevance Model {#section-3}

**Architecture**: DistilBERT + multi-label classification head
- Output: 85 independent sigmoid units (one per category)
- Loss: BCEWithLogitsLoss (binary cross-entropy, NOT CrossEntropyLoss)
- Each label is treated as an independent binary classification problem


In [ ]:
# === Multi-Label Dataset ===

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

class MultiLabelDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx] if hasattr(self.texts, 'iloc') else self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.float32)  # float for BCE
        }

# Create datasets
train_dataset = MultiLabelDataset(X_train_df['description_clean'], y_train, tokenizer)
val_dataset = MultiLabelDataset(X_val_df['description_clean'], y_val, tokenizer)
test_dataset = MultiLabelDataset(X_test_df['description_clean'], y_test, tokenizer)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=0, pin_memory=(device.type == 'cuda'))
val_loader = DataLoader(val_dataset, batch_size=batch_size,
                        num_workers=0, pin_memory=(device.type == 'cuda'))
test_loader = DataLoader(test_dataset, batch_size=batch_size,
                         num_workers=0, pin_memory=(device.type == 'cuda'))

print(f'DataLoaders created:')
print(f'  Train: {len(train_loader)} batches ({len(train_dataset):,} samples)')
print(f'  Val:   {len(val_loader)} batches ({len(val_dataset):,} samples)')
print(f'  Test:  {len(test_loader)} batches ({len(test_dataset):,} samples)')
print(f'  Batch size: {batch_size}')


In [ ]:
# === BERT Binary Relevance Model ===

class DistilBERTMultiLabel(nn.Module):
    """
    DistilBERT + multi-label classification head.
    Each of the 85 outputs is an independent sigmoid (Binary Relevance).
    Trained with BCEWithLogitsLoss.
    """
    def __init__(self, num_labels, dropout_rate=0.2):
        super().__init__()
        self.distilbert = AutoModel.from_pretrained('distilbert-base-uncased')
        self.pre_classifier = nn.Linear(self.distilbert.config.hidden_size, self.distilbert.config.hidden_size)
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.distilbert.config.hidden_size, num_labels)
        self.relu = nn.ReLU()
    
    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]  # [CLS] token
        pooled = self.relu(self.pre_classifier(pooled))
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)  # Raw logits (no sigmoid here)
        return logits

model = DistilBERTMultiLabel(num_labels=num_labels).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: DistilBERT Multi-Label (Binary Relevance)')
print(f'  Output units: {num_labels} (independent sigmoids)')
print(f'  Total parameters: {total_params:,}')
print(f'  Loss: BCEWithLogitsLoss')
print(f'  Device: {device}')


# SECTION 4: Training Loop (Multi-Label) {#section-4}

Key differences from single-label training:
- **Loss**: BCEWithLogitsLoss (not CrossEntropyLoss)
- **Predictions**: sigmoid(logits) > threshold (not argmax)
- **Metrics**: Micro-F1, Macro-F1, LRAP, Hamming Loss (not accuracy)


In [ ]:
# === Multi-Label Training Function ===

def train_multilabel(model, train_loader, val_loader, criterion, optimizer, scheduler,
                     device, num_epochs=10, patience=3, model_name='bert_br', threshold=0.5):
    """Training loop for multi-label classification with early stopping on Micro-F1."""
    
    best_val_micro_f1 = 0
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_micro_f1': [], 'val_macro_f1': []}
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # === Training ===
        model.train()
        total_train_loss = 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_train_loss += loss.item()
        
        avg_train_loss = total_train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)
        
        # === Validation ===
        model.eval()
        total_val_loss = 0
        all_logits = []
        all_labels = []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)
                total_val_loss += loss.item()
                
                all_logits.append(logits.cpu())
                all_labels.append(labels.cpu())
        
        avg_val_loss = total_val_loss / len(val_loader)
        history['val_loss'].append(avg_val_loss)
        
        # Compute metrics
        all_logits = torch.cat(all_logits, dim=0)
        all_labels = torch.cat(all_labels, dim=0).numpy()
        probs = torch.sigmoid(all_logits).numpy()
        preds = (probs > threshold).astype(int)
        
        val_micro_f1 = f1_score(all_labels, preds, average='micro', zero_division=0)
        val_macro_f1 = f1_score(all_labels, preds, average='macro', zero_division=0)
        history['val_micro_f1'].append(val_micro_f1)
        history['val_macro_f1'].append(val_macro_f1)
        
        print(f'Epoch {epoch+1:2d}/{num_epochs} | '
              f'Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | '
              f'Micro-F1: {val_micro_f1:.4f} | Macro-F1: {val_macro_f1:.4f}', end='')
        
        # Early stopping on Micro-F1
        if val_micro_f1 > best_val_micro_f1:
            best_val_micro_f1 = val_micro_f1
            patience_counter = 0
            torch.save(model.state_dict(), f'best_{model_name}.pt')
            print(f' -> BEST')
        else:
            patience_counter += 1
            print(f' ({patience_counter}/{patience})')
        
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break
    
    total_time = time.time() - start_time
    print(f'\nTraining complete: {total_time:.0f}s ({total_time/60:.1f} min)')
    print(f'Best Val Micro-F1: {best_val_micro_f1:.4f}')
    
    # Load best model
    model.load_state_dict(torch.load(f'best_{model_name}.pt'))
    
    return history, best_val_micro_f1

print('Multi-label training function defined')


In [ ]:
# === Train BERT Binary Relevance ===

print('='*80)
print('TRAINING: DistilBERT Binary Relevance (Categories)')
print('='*80)
print()

# Loss: BCEWithLogitsLoss (combines sigmoid + BCE, numerically stable)
criterion = nn.BCEWithLogitsLoss(reduction='mean')

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

# Scheduler
num_training_steps = len(train_loader) * 10
scheduler = CosineAnnealingLR(optimizer, T_max=num_training_steps, eta_min=1e-6)

# Train
history, best_val_f1 = train_multilabel(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, num_epochs=10, patience=3, model_name='bert_br_categories'
)


# SECTION 5: Threshold Optimization {#section-5}

The default threshold of 0.5 is rarely optimal for multi-label classification.
Optimize globally on validation set to maximize Micro-F1.

**Critical**: Thresholds are tuned on VALIDATION set only. Test set is untouched.


In [ ]:
# === Threshold Optimization on Validation Set ===

print('='*80)
print('THRESHOLD OPTIMIZATION (Validation Set)')
print('='*80)
print()

# Get validation logits from best model
model.eval()
val_logits_list = []
val_labels_list = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']
        
        logits = model(input_ids, attention_mask)
        val_logits_list.append(logits.cpu())
        val_labels_list.append(labels)

val_logits_all = torch.cat(val_logits_list, dim=0)
val_labels_all = torch.cat(val_labels_list, dim=0).numpy()
val_probs_all = torch.sigmoid(val_logits_all).numpy()

# === Global threshold search ===
thresholds = np.arange(0.1, 0.7, 0.025)
threshold_results = []

for t in thresholds:
    preds = (val_probs_all > t).astype(int)
    micro = f1_score(val_labels_all, preds, average='micro', zero_division=0)
    macro = f1_score(val_labels_all, preds, average='macro', zero_division=0)
    avg_preds = preds.sum(axis=1).mean()
    threshold_results.append({'threshold': t, 'micro_f1': micro, 'macro_f1': macro, 'avg_preds': avg_preds})

threshold_df = pd.DataFrame(threshold_results)
best_thresh_idx = threshold_df['micro_f1'].idxmax()
best_threshold = threshold_df.loc[best_thresh_idx, 'threshold']
best_micro_at_thresh = threshold_df.loc[best_thresh_idx, 'micro_f1']
best_macro_at_thresh = threshold_df.loc[best_thresh_idx, 'macro_f1']

print(f'Threshold search results:')
print(threshold_df.to_string(index=False))
print()
print(f'Optimal threshold: {best_threshold:.3f}')
print(f'  Micro-F1 at optimal: {best_micro_at_thresh:.4f}')
print(f'  Macro-F1 at optimal: {best_macro_at_thresh:.4f}')
print(f'  Micro-F1 at 0.5:     {threshold_df.loc[threshold_df["threshold"].sub(0.5).abs().idxmin(), "micro_f1"]:.4f}')
print()

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(threshold_df['threshold'], threshold_df['micro_f1'], 'b-o', label='Micro-F1', linewidth=2)
ax.plot(threshold_df['threshold'], threshold_df['macro_f1'], 'r-o', label='Macro-F1', linewidth=2)
ax.axvline(x=best_threshold, color='green', linestyle='--', label=f'Optimal: {best_threshold:.3f}')
ax.axvline(x=0.5, color='gray', linestyle=':', label='Default (0.5)')
ax.set_xlabel('Threshold', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Threshold Optimization (Validation Set)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_optimization_categories.png', dpi=100, bbox_inches='tight')
plt.show()


# SECTION 6: Results & Per-Label Analysis {#section-6}

Final evaluation on held-out test set using optimized threshold.
Compare BERT-BR against TF-IDF baseline.


In [ ]:
# === Final Test Evaluation with Optimal Threshold ===

print('='*80)
print('FINAL TEST EVALUATION')
print('='*80)
print()

model.eval()
test_logits_list = []
test_labels_list = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']
        
        logits = model(input_ids, attention_mask)
        test_logits_list.append(logits.cpu())
        test_labels_list.append(labels)

test_logits_all = torch.cat(test_logits_list, dim=0)
test_labels_all = torch.cat(test_labels_list, dim=0).numpy()
test_probs_all = torch.sigmoid(test_logits_all).numpy()

# Apply optimal threshold
test_preds_opt = (test_probs_all > best_threshold).astype(int)
test_preds_05 = (test_probs_all > 0.5).astype(int)

# === Metrics with optimal threshold ===
from sklearn.metrics import hamming_loss, label_ranking_average_precision_score

test_micro_f1 = f1_score(test_labels_all, test_preds_opt, average='micro', zero_division=0)
test_macro_f1 = f1_score(test_labels_all, test_preds_opt, average='macro', zero_division=0)
test_hamming = hamming_loss(test_labels_all, test_preds_opt)
test_subset_acc = (test_labels_all == test_preds_opt).all(axis=1).mean()
test_lrap = label_ranking_average_precision_score(test_labels_all, test_probs_all)

print(f'BERT-BR with optimal threshold ({best_threshold:.3f}):')
print(f'  Micro-F1:      {test_micro_f1:.4f}')
print(f'  Macro-F1:      {test_macro_f1:.4f}')
print(f'  Hamming Loss:  {test_hamming:.4f}')
print(f'  Subset Acc:    {test_subset_acc:.4f}')
print(f'  LRAP:          {test_lrap:.4f}')
print()

# === Comparison: BERT-BR vs TF-IDF ===
print('MODEL COMPARISON (Test Set):')
comparison = pd.DataFrame({
    'Metric': ['Micro-F1', 'Macro-F1', 'Hamming Loss', 'Subset Acc', 'LRAP'],
    'TF-IDF BR': [tfidf_results['micro_f1'], tfidf_results['macro_f1'],
                  tfidf_results['hamming'], tfidf_results['subset_acc'], tfidf_results['lrap']],
    'BERT-BR': [test_micro_f1, test_macro_f1, test_hamming, test_subset_acc, test_lrap],
})
comparison['Delta'] = comparison['BERT-BR'] - comparison['TF-IDF BR']
comparison['Delta'] = comparison.apply(
    lambda r: -r['Delta'] if r['Metric'] == 'Hamming Loss' else r['Delta'], axis=1
)
print()
print(comparison.to_string(index=False))
print()

# Store BERT results
bert_br_results = {
    'micro_f1': test_micro_f1,
    'macro_f1': test_macro_f1,
    'hamming': test_hamming,
    'subset_acc': test_subset_acc,
    'lrap': test_lrap,
    'threshold': best_threshold,
    'test_preds': test_preds_opt,
    'test_probs': test_probs_all,
    'test_labels': test_labels_all
}


In [ ]:
# === Per-Label F1 Analysis ===

print('='*80)
print('PER-LABEL F1 PERFORMANCE (Test Set)')
print('='*80)
print()

per_label_f1 = []
for i, label in enumerate(label_names):
    f1 = f1_score(test_labels_all[:, i], test_preds_opt[:, i], zero_division=0)
    support = test_labels_all[:, i].sum()
    per_label_f1.append({'Label': label, 'F1': f1, 'Support': int(support)})

per_label_df = pd.DataFrame(per_label_f1).sort_values('F1', ascending=False).reset_index(drop=True)

print('Top 10 best-performing labels:')
print(per_label_df.head(10).to_string(index=False))
print()
print('Bottom 10 worst-performing labels:')
print(per_label_df.tail(10).to_string(index=False))
print()

# Summary stats
print(f'Per-label F1 summary:')
print(f'  Mean:   {per_label_df["F1"].mean():.4f}')
print(f'  Median: {per_label_df["F1"].median():.4f}')
print(f'  Std:    {per_label_df["F1"].std():.4f}')
print(f'  Labels with F1 = 0: {(per_label_df["F1"] == 0).sum()}')
print(f'  Labels with F1 > 0.5: {(per_label_df["F1"] > 0.5).sum()}')
print(f'  Labels with F1 > 0.7: {(per_label_df["F1"] > 0.7).sum()}')

# === F1 vs Support correlation (RQ3) ===
from scipy.stats import spearmanr

corr, pval = spearmanr(per_label_df['Support'], per_label_df['F1'])
print(f'\nCorrelation (F1 vs Support): Spearman r={corr:.3f}, p={pval:.4f}')
if pval < 0.05:
    print(f'  Significant positive correlation: more training examples -> higher F1')


# SECTION 7: Error Analysis & Visualization {#section-7}

Systematic error analysis to understand model failures and inform future work.


In [ ]:
# === Visualization: Per-Label F1 vs Support ===

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Per-label F1 distribution
axes[0].hist(per_label_df['F1'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(per_label_df['F1'].mean(), color='red', linestyle='--', label=f'Mean: {per_label_df["F1"].mean():.3f}')
axes[0].axvline(per_label_df['F1'].median(), color='orange', linestyle='--', label=f'Median: {per_label_df["F1"].median():.3f}')
axes[0].set_xlabel('F1 Score', fontsize=12)
axes[0].set_ylabel('Number of Labels', fontsize=12)
axes[0].set_title('Distribution of Per-Label F1 Scores', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)

# Plot 2: F1 vs Support (log scale)
axes[1].scatter(per_label_df['Support'], per_label_df['F1'], alpha=0.6, s=40, color='steelblue')
axes[1].set_xscale('log')
axes[1].set_xlabel('Label Support (log scale)', fontsize=12)
axes[1].set_ylabel('F1 Score', fontsize=12)
axes[1].set_title(f'F1 vs Label Frequency (Spearman r={corr:.3f})', fontsize=13, fontweight='bold')

# Annotate worst labels
for _, row in per_label_df.tail(5).iterrows():
    axes[1].annotate(row['Label'], (row['Support'], row['F1']), fontsize=8, alpha=0.7)

plt.tight_layout()
plt.savefig('per_label_analysis_categories.png', dpi=100, bbox_inches='tight')
plt.show()
print('Per-label analysis plot saved')


In [ ]:
# === Training Progress Visualization ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss curves
axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], 'r-o', label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('BCE Loss', fontsize=12)
axes[0].set_title('Training Progress: Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# F1 curves
axes[1].plot(epochs_range, history['val_micro_f1'], 'g-o', label='Val Micro-F1', linewidth=2)
axes[1].plot(epochs_range, history['val_macro_f1'], 'm-o', label='Val Macro-F1', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('F1 Score', fontsize=12)
axes[1].set_title('Validation Performance: Multi-Label F1', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_progress_categories.png', dpi=100, bbox_inches='tight')
plt.show()
print('Training progress plot saved')


In [ ]:
# === Final Phase 3 Report ===

print('='*80)
print('PHASE 3: CATEGORY CLASSIFICATION - FINAL REPORT')
print('='*80)
print()

print(f'TASK: Multi-Label Classification (85 categories)')
print(f'DATASET: {len(df):,} games, avg {y_multilabel.sum(axis=1).mean():.2f} labels/game')
print(f'IMBALANCE: ratio {y_multilabel.sum(axis=0).max()}/{y_multilabel.sum(axis=0).min()} = {y_multilabel.sum(axis=0).max()/y_multilabel.sum(axis=0).min():.0f}x')
print()

print('RESULTS (Test Set):')
print(f'  {"Model":<20} {"Micro-F1":>10} {"Macro-F1":>10} {"LRAP":>10} {"Hamming":>10}')
print(f'  {"-"*60}')
print(f'  {"TF-IDF BR":<20} {tfidf_results["micro_f1"]:>10.4f} {tfidf_results["macro_f1"]:>10.4f} {tfidf_results["lrap"]:>10.4f} {tfidf_results["hamming"]:>10.4f}')
print(f'  {"BERT-BR":<20} {test_micro_f1:>10.4f} {test_macro_f1:>10.4f} {test_lrap:>10.4f} {test_hamming:>10.4f}')
print()

print(f'THRESHOLD: {best_threshold:.3f} (optimized on validation set)')
print()

print(f'PER-LABEL ANALYSIS:')
print(f'  Labels with F1 > 0.5: {(per_label_df["F1"] > 0.5).sum()}/{num_labels}')
print(f'  Labels with F1 = 0.0: {(per_label_df["F1"] == 0).sum()}/{num_labels}')
print(f'  F1-Support correlation: r={corr:.3f} (p={pval:.4f})')
print()

print('ARTIFACTS SAVED:')
print('  - best_bert_br_categories.pt (model checkpoint)')
print('  - threshold_optimization_categories.png')
print('  - per_label_analysis_categories.png')
print('  - training_progress_categories.png')
print()

print('NEXT STEPS:')
print('  - Apply same pipeline to mechanisms (195 labels)')
print('  - Implement MLGN architecture for RQ2 comparison')
print('  - Statistical validation (McNemar, Bootstrap CI)')
print('='*80)

# Save results
results_phase3 = {
    'tfidf': tfidf_results,
    'bert_br': {
        'micro_f1': float(test_micro_f1),
        'macro_f1': float(test_macro_f1),
        'hamming': float(test_hamming),
        'subset_acc': float(test_subset_acc),
        'lrap': float(test_lrap),
        'threshold': float(best_threshold),
    },
    'per_label_f1': per_label_df[['Label', 'F1', 'Support']].to_dict('records'),
    'num_labels': num_labels,
    'dataset_size': len(df),
}

with open('phase3_categories_results.json', 'w') as f:
    json.dump(results_phase3, f, indent=2)

print('Results saved to phase3_categories_results.json')
